In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np
import networkx as nx

# FUNCTIONS

In [4]:
# define a function to load a pickle
def load_pickle(file_name):
    if os.path.exists(file_name):
        with open(file_name, 'rb') as handle:
            de_pickle = pickle.load(handle)
    else:
        de_pickle = None
        print("file does not exist")
    return de_pickle

In [5]:
def get_vowels(letters, r_word, min_vowel_count, existing_words):    

    # combine the existing letters and the remaining word
    letter_group = set(letters + r_word)
    rg_vowels = vowel_set.difference(letter_group)
    rg_vowel_count = len(rg_vowels)    

    if rg_vowel_count >= min_vowel_count:        
        letter_group = ''.join(sorted(letter_group))    
        remainder_group  = lc_set.difference(letter_group)
        remainder_group = ''.join(sorted(remainder_group))
        temp_list = [r_word, letter_group, remainder_group]
        existing_words.extend(temp_list)
        return existing_words
    else:
        return None

# LOAD LETTERS

In [6]:
letter_dict = load_pickle(file_name = 'letter_dict.pkl')

In [7]:
word_df = pd.read_csv(filepath_or_buffer=  'words_alpha.txt', header = None, names = ['word'], dtype = str)

In [8]:
word_df['word'] = word_df['word'].astype(str)

In [9]:
word_df.head()

,word
0,a
1,aa
2,aaa
3,aah
4,aahed


In [10]:
word_df.shape

(370105, 1)

In [11]:
word_df['word'].isna().value_counts()

word
False    370105
Name: count, dtype: int64

In [12]:
word_df['lcase'] = word_df['word'].str.lower()

In [13]:
word_df['n_letters'] = word_df['word'].str.len()

In [14]:
word_df['letters_sorted'] = word_df['lcase'].map(lambda x: ''.join(sorted(x)))

In [15]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))
word_df['lcase_tuple'] = word_df['lcase_set'].map(lambda x: tuple(x))

In [16]:
word_df['n_unique_chars'] = word_df['lcase_set'].map(lambda x: len(x))

In [17]:
word_df = word_df.loc[(word_df['n_unique_chars'] == 5) & (word_df['n_letters'] == 5), :]
word_df = word_df.sort_values(by = 'lcase').reset_index(drop = True)

In [18]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['letters_sorted'])

In [19]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,lcase_tuple,n_unique_chars
0,abdom,abdom,5,abdmo,"{a, m, o, d, b}","(a, m, o, d, b)",5
1,abend,abend,5,abden,"{a, n, e, d, b}","(a, n, e, d, b)",5
2,abets,abets,5,abest,"{a, e, t, s, b}","(a, e, t, s, b)",5
3,abhor,abhor,5,abhor,"{a, r, h, o, b}","(a, r, h, o, b)",5
4,abide,abide,5,abdei,"{a, i, e, d, b}","(a, i, e, d, b)",5


In [20]:
word_df.shape

(5977, 7)

In [21]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


In [22]:
word_df['word_id'] = range(0, word_df.shape[0])

In [23]:
word_df.shape

(5977, 8)

In [24]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,lcase_tuple,n_unique_chars,word_id
0,abdom,abdom,5,abdmo,"{a, m, o, d, b}","(a, m, o, d, b)",5,0
1,abend,abend,5,abden,"{a, n, e, d, b}","(a, n, e, d, b)",5,1
2,abets,abets,5,abest,"{a, e, t, s, b}","(a, e, t, s, b)",5,2
3,abhor,abhor,5,abhor,"{a, r, h, o, b}","(a, r, h, o, b)",5,3
4,abide,abide,5,abdei,"{a, i, e, d, b}","(a, i, e, d, b)",5,4


In [25]:
word_id_list = word_df['word_id'].to_numpy(dtype = np.int16)

In [26]:
working_word_df = word_df[['word_id', 'lcase']].copy()

In [27]:
# build dictionaries
word_dict = {}
for i_row, my_row in word_df.iterrows():
    word_dict[my_row['lcase']] = my_row['lcase_set']

In [28]:
lc_set = set(ascii_lowercase)

In [29]:
vowel_set = set('aeiouy')

# BUILD A CHAR MATRIX

In [30]:
# build a char_matrix
char_matrix = np.zeros(shape = (word_df.shape[0], 26), dtype = np.int8)
def build_char_matrix(row):
    word = sorted(row['lcase'])
    for iw, w in enumerate(word):
        char_matrix[row['word_id'], letter_dict[w]] += 1


In [31]:
outcome = word_df.apply(build_char_matrix, axis = 1)

In [32]:
assert word_df.shape[0] == char_matrix.shape[0]

In [33]:
char_matrix.shape

(5977, 26)

# BUILD LEVEL 2 USING COMBINATIONS

In [34]:
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    w1w2_set = word_dict[lc0].union(word_dict[lc1])    
    if len(w1w2_set) == 10:

        # compute the number of vowels left over
        remaining_vowels = vowel_set.difference(w1w2_set)
        # need at least three vowels - for three separate words to continue
        if len(remaining_vowels) >= 3:
            
            existing_letters = ''.join(sorted(w1w2_set))
            remainder_letters = ''.join(sorted(lc_set.difference(existing_letters)))
            
            edge_list.append([lc0, lc1, existing_letters, remainder_letters ])
            

In [35]:
# export to an edgelist
l2_df = pd.DataFrame(data = edge_list, columns = ['w1', 'w2', 'l2_existing', 'r2_remainder'])

In [36]:
l2_df.shape

(1186081, 4)

In [37]:
l2_df = l2_df.drop_duplicates(subset = ['l2_existing', 'r2_remainder'])

In [38]:
l2_df.shape

(318855, 4)

In [39]:
l2_df.to_csv(path_or_buf='level2.txt', sep = '\t', index = False)

In [40]:
l2_df.head()

,w1,w2,l2_existing,r2_remainder
0,abdom,celts,abcdelmost,fghijknpqruvwxyz
1,abdom,cents,abcdemnost,fghijklpqruvwxyz
2,abdom,chefs,abcdefhmos,gijklnpqrtuvwxyz
3,abdom,chelp,abcdehlmop,fgijknqrstuvwxyz
4,abdom,cheng,abcdeghmno,fijklpqrstuvwxyz


In [41]:
l2_df.shape

(318855, 4)

In [42]:
l2_df['l2_existing'].unique().shape

(318855,)

# BUILD LEVEL 3 USING ENUMERATION

In [43]:
test_l2_df = l2_df.iloc[:1000]

In [44]:
test_l2_df.head()

,w1,w2,l2_existing,r2_remainder
0,abdom,celts,abcdelmost,fghijknpqruvwxyz
1,abdom,cents,abcdemnost,fghijklpqruvwxyz
2,abdom,chefs,abcdefhmos,gijklnpqrtuvwxyz
3,abdom,chelp,abcdehlmop,fgijknqrstuvwxyz
4,abdom,cheng,abcdeghmno,fijklpqrstuvwxyz


In [148]:
l3_df_list = []
candidate_words_list = []
for i_row, row in test_l2_df.iterrows():
    # this is really three rounds of enumeration - all slow
    
    w1, w2, l2, r2 = row

    # these are the words with the existing letters
    #existing_id_list = []
    l_idx_list = [letter_dict[l] for l in l2]    
    test_char_matrix = char_matrix[:, l_idx_list] == 1        
    existing_id_list = word_id_list[test_char_matrix.any(axis = 1)]    
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)    

    # these are the word ids of candidates 
    if remainder_id_list.size > 0:         
        n_candidate_words = remainder_id_list.shape[0]               
    
        # get the remainder ids from the char_matrix - sub matrix
        test_match = char_matrix[remainder_id_list, :]
        # now, filter out records that do not already have the existing letters
        test_match = (test_match[:, l_idx_list] == 0).all(axis = 1)

        candidate_word_ids = remainder_id_list[test_match]
        if candidate_word_ids.size > 0:            

            l3_df = working_word_df.loc[working_word_df['word_id'].isin(candidate_word_ids), :].rename(columns = {'lcase':'w3'})
            l3_df['w1'] = w1
            l3_df['w2'] = w2
            
            #print(len(output))
            l3_df_list.append(l3_df)




In [153]:
l3_df = pd.concat(objs = l3_df_list)

In [154]:
l3_df['l3'] = (l3_df['w3'] + l2).map(lambda x: ''.join(sorted(set(x))))
l3_df['r3'] = (l3_df['l3'].map(lambda x: ''.join(sorted(lc_set.difference(x)))))
            

In [156]:
l3_df.shape

(93169, 6)

In [157]:
l3_df.head()

,word_id,w3,w1,w2,l3,r3
3114,2300,fingu,abdom,celts,abfghikmnorsu,cdejlpqtvwxyz
3135,2318,fixup,abdom,celts,abfhikmnoprsux,cdegjlqtvwyz
3420,2541,funky,abdom,celts,abfhikmnorsuy,cdegjlpqtvwxz
3429,2549,furzy,abdom,celts,abfhikmnorsuyz,cdegjlpqtvwx
3614,2679,girny,abdom,celts,abghikmnorsy,cdefjlpqtuvwxz


In [141]:
l3_df_test = l3_df_test.drop_duplicates(subset=['l3', 'r3'])

In [142]:
l3_df_test.shape

(26618, 6)

In [143]:
l3_df_test.head()

,word_id,w3,w1,w2,l3,r3
3114,2300,fingu,abdom,celts,abcdefgilmnostu,hjkpqrvwxyz
3135,2318,fixup,abdom,celts,abcdefilmopstux,ghjknqrvwyz
3420,2541,funky,abdom,celts,abcdefklmnostuy,ghijpqrvwxz
3429,2549,furzy,abdom,celts,abcdeflmorstuyz,ghijknpqvwx
3614,2679,girny,abdom,celts,abcdegilmnorsty,fhjkpquvwxz


In [80]:
char_matrix.shape

(5977, 26)

In [75]:
remainder_id_list

array([2300, 2318, 2541, 2549, 2679, 2811, 2814, 2837, 2874, 3130, 3143,
       3233, 3330, 3405, 3501, 3514, 3591, 4510, 4518, 4635, 4638, 4683,
       4685, 4690, 4699, 4755, 4762, 5468, 5473, 5476, 5505, 5507, 5508,
       5660, 5759, 5761, 5795, 5843, 5847, 5952, 5953], dtype=int16)

In [76]:
l_idx_list

[0, 1, 2, 3, 4, 11, 12, 14, 18, 19]

In [131]:
l3_df_list = []
candidate_words_list = []
for i_row, row in test_l2_df.iterrows():
    # this is really three rounds of enumeration - all slow
    
    w1, w2, l2, r2 = row

    # these are the words with the existing letters
    #existing_id_list = []
    l_idx_list = [letter_dict[l] for l in l2]        
    test_char_matrix = char_matrix[:, l_idx_list] == 1        
    existing_id_list = word_id_list[test_char_matrix.any(axis = 1)]    
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)    

    # these are the word ids of candidates 
    if remainder_id_list.size > 0:         
        n_candidate_words = remainder_id_list.shape[0]               
    
        # how can we winnow down the list of candidates?
        r3_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()          
        temp_list = [w1, w2, l2, r2, n_candidate_words]
        candidate_words_list.append(temp_list)
        for r3_word in r3_word_list:            
            ex_words = [w1, w2]
            output = get_vowels(letters=l2, r_word = r3_word, min_vowel_count=2, existing_words=ex_words)            
            
            if output:                
                #print(len(output))
                l3_df_list.append(output)




In [123]:
l3_df = pd.DataFrame(data = l3_df_list, columns = ['w1', 'w2', 'w3', 'l3', 'r3'])

In [124]:
l3_df.shape

(24222, 5)

In [125]:
l3_df.head()

,w1,w2,w3,l3,r3
0,abdom,celts,griph,abcdeghilmoprst,fjknquvwxyz
1,abdom,celts,gryph,abcdeghlmoprsty,fijknquvwxz
2,abdom,celts,prink,abcdeiklmnoprst,fghjquvwxyz
3,abdom,celts,whing,abcdeghilmnostw,fjkpqruvxyz
4,abdom,celts,wring,abcdegilmnorstw,fhjkpquvxyz


In [ ]:
l3_df.shape

In [126]:
l3_df = l3_df.drop_duplicates(subset = ['l3', 'r3'])

In [127]:
l3_df.shape

(5088, 5)

In [134]:
cw_df = pd.DataFrame(data = candidate_words_list, columns = ['w1', 'w2', 'l2', 'rr2', 'n_words'])

In [135]:
cw_df.head()

,w1,w2,l2,rr2,n_words
0,abdom,celts,abcdelmost,fghijknpqruvwxyz,41
1,abdom,cents,abcdemnost,fghijklpqruvwxyz,35
2,abdom,chefs,abcdefhmos,gijklnpqrtuvwxyz,99
3,abdom,chelp,abcdehlmop,fgijknqrstuvwxyz,136
4,abdom,cheng,abcdeghmno,fijklpqrstuvwxyz,118


In [ ]:
test_df.to_csv(path_or_buf='level3.txt', sep = '\t', index = False)

In [ ]:
test_df.head()

# BUILD LEVEL 4 USING ENUMERATION

In [ ]:
l4_df_list = []
for i_row, row in test_df.iterrows():

    w1, w2, w3, l3, r3 = row

    # existing letters
    existing_id_list = []
    for l in l3:
        l_idx = letter_dict[l]   
        test_char_matrix = char_matrix[:, l_idx] == 1
        # print(word_id_list[test_char_matrix])
        existing_id_list.append(word_id_list[test_char_matrix])

    #print(inclusive_id_list)
    existing_id_list = np.concatenate(existing_id_list)        
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)
    if remainder_id_list.size > 0:                
        #print(l1, r1, outcome_ids.shape)        
        #level2_list.append([l1, r1, existing_id_list.shape[0], remainder_id_list.shape[0]])                       
        #         
        # l1
        # do this later - winnow down the set list sooner
        # do this later - winnow down the set list sooner
        r4_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()
        # print(r3_word_list)
        for r4_word in r4_word_list:
            output = get_vowels(letters=l3, r_word = r4_word, min_vowel_count=1, existing_words=[w1, w2, w3])
            if output:
                l4_df_list.append(output)

In [ ]:
l4_df = pd.DataFrame(data = l4_df_list, columns =  ['w1', 'w2', 'w3', 'w4', 'l4', 'r4'])

In [ ]:
l4_df.shape

In [ ]:
l4_df.head()

In [ ]:
test_df = l4_df.drop_duplicates(subset = ['l4', 'r4'])

In [ ]:
test_df.shape

In [ ]:
test_df.to_csv(path_or_buf='level4.txt', index = False, sep =  '\t')

# BUILD LEVEL 5 USING ENUMERATIONS

In [ ]:
l5_df_list = []
for i_row, row in test_df.iterrows():
    #print(i_row)

    w1, w2, w3, w4, l4, r4 = row

    # existing letters
    existing_id_list = []
    for l in l4:
        l_idx = letter_dict[l]   
        test_char_matrix = char_matrix[:, l_idx] == 1
        # print(word_id_list[test_char_matrix])
        existing_id_list.append(word_id_list[test_char_matrix])

    #print(inclusive_id_list)
    existing_id_list = np.concatenate(existing_id_list)        
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)
    if remainder_id_list.size > 0:                
        #print(l1, r1, outcome_ids.shape)        
        #level2_list.append([l1, r1, existing_id_list.shape[0], remainder_id_list.shape[0]])                       
        #         
        # l1
        # do this later - winnow down the set list sooner
        # do this later - winnow down the set list sooner
        r5_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()
        # print(r3_word_list)
        for r5_word in r5_word_list:
            output = get_vowels(letters=l4, r_word = r5_word, min_vowel_count=0, existing_words=[w1, w2, w3, w4])
            if output:
                l5_df_list.append(output)

In [ ]:
l5_df = pd.DataFrame(data = l5_df_list, columns =  ['w1', 'w2', 'w3', 'w4', 'w5', 'l5', 'r5'])

In [ ]:
l5_df.head()

In [ ]:
l5_df.shape

In [ ]:
l5_df = l5_df.drop_duplicates(subset = ['l5', 'r5']).reset_index(drop = True)

In [ ]:
l5_df.head()

In [ ]:
l5_df

In [ ]:
l5_df.to_csv(path_or_buf='level5.txt', sep = '\t', index = False)